In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/gods/gods/train.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df['content'].fillna('', inplace=True)
df['text'] = df['title'] + " " + df['content']
df.info()

<ipython-input-16-241eaa995a7a>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['content'].fillna('', inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22151 entries, 0 to 22150
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       22151 non-null  int64 
 1   title    22151 non-null  object
 2   content  22151 non-null  object
 3   target   22151 non-null  object
 4   text     22151 non-null  object
dtypes: int64(1), object(4)
memory usage: 865.4+ KB


In [ ]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    return text

df['clean_text'] = df['text'].apply(clean_text)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['target']


In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 5.0 MB/s eta 0:00:00


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

from catboost import CatBoostClassifier

# Update CatBoost parameters
models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "CatBoost": CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=10,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    task_type="CPU",  # Force CPU mode to avoid GPU issues
    verbose=50
    ),


}


# Train models and evaluate accuracy
accuracies = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    accuracies[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

# Select the best model
best_model_name = max(accuracies, key=accuracies.get)
best_model = models[best_model_name]

print(f"\nBest Model: {best_model_name} with Accuracy: {accuracies[best_model_name]:.4f}")


Logistic Regression Accuracy: 0.7547
0:	learn: 1.5433568	total: 9.51s	remaining: 47m 23s
50:	learn: 0.8936504	total: 4m 36s	remaining: 22m 29s
100:	learn: 0.8077982	total: 8m 54s	remaining: 17m 32s
150:	learn: 0.7653082	total: 13m 11s	remaining: 13m 1s
200:	learn: 0.7338180	total: 17m 29s	remaining: 8m 36s
250:	learn: 0.7031547	total: 21m 45s	remaining: 4m 14s
299:	learn: 0.6812515	total: 25m 56s	remaining: 0us
CatBoost Accuracy: 0.7321

Best Model: Logistic Regression with Accuracy: 0.7547


In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

# Create Soft Voting Classifier with best models
ensemble_model = VotingClassifier(
    estimators=[(name, model) for name, model in best_models.items()],
    voting='soft'  # Soft Voting (average probabilities)
)

# Train ensemble model
ensemble_model.fit(X_train, y_train)

# Evaluate on validation set
y_pred_ensemble = ensemble_model.predict(X_val)
ensemble_acc = accuracy_score(y_val, y_pred_ensemble)
print("\nSoft Voting Ensemble Accuracy:", ensemble_acc)



Soft Voting Ensemble Accuracy: 0.7546829158203566


In [ ]:
test_df = pd.read_csv('/content/drive/My Drive/gods/gods/test.csv')
test_df['content'].fillna('', inplace=True)
test_df['text'] = test_df['title'] + " " + test_df['content']
test_df['clean_text'] = test_df['text'].apply(clean_text)

X_test_final = vectorizer.transform(test_df['clean_text'])

test_df['target'] = best_model.predict(X_test_final)

submission = test_df[['id', 'target']]
submission.to_csv('submission.csv', index=False)


<ipython-input-34-1bda1bd2072c>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['content'].fillna('', inplace=True)
